# ప్రాధాన్య సభ్య మిడిల్‌వేర్‌తో హోటల్ బుకింగ్

ఈ నోట్బుక్ Microsoft Agent Framework ఉపయోగించి **ఫంక్షన్-ఆధారిత మిడిల్‌వేర్** ను చూపిస్తుంది. ప్రాధాన్య సభ్యులకు ప్రత్యేక అనుమతులు ఇచ్చే మిడిల్‌వేర్ పొరను జోడించి కండిషనల్ వర్క్‌ఫ్లో ఉదాహరణపై నిర్మాణం చేశాము.

## మీరు నేర్చుకునేది:
1. **ఫంక్షన్-ఆధారిత మిడిల్‌వేర్**: ఫంక్షన్ ఫలితాలను ఫిర్యాదుచేసి సవరించడం
2. **కాంటెక్స్ యాక్సెస్**: ఎక్జిక్యూషన్ తర్వాత `context.result` చదవడం మరియు సవరించడం
3. **బిజినెస్ లాజిక్ అమలు**: ప్రాధాన్య సభ్యుల ప్రయోజనాలు
4. **ఫలితాన్ని ఓవర్రైడ్ చేయడం**: వినియోగదారుల స్థితి ఆధారంగా ఫంక్షన్ ఫలితాలను మార్చడం
5. **ఒకే వర్క్‌ఫ్లో, వేరే ఫలితాలు**: మిడిల్‌వేర్ ఆధారిత ప్రవర్తన మార్పులు

## మిడిల్‌వేర్తో వర్క్‌ఫ్లో ఆర్కిటెక్చర్:

```
User Input: "I want to book a hotel in Paris"
                    ↓
        [availability_agent]
        - Calls hotel_booking tool
        - 🌟 priority_check middleware intercepts
        - Checks user membership status
        - IF priority + no rooms → Override to available!
        - Returns BookingCheckResult
                    ↓
        Conditional Routing
           /                    \
    [has_availability]    [no_availability]
          ↓                      ↓
    [booking_agent]        [alternative_agent]
    (Priority override!)   (Regular users)
          ↓                      ↓
       [display_result executor]
```

## కండిషనల్ వర్క్‌ఫ్లో నుండి ప్రధాన తేడా:

**మిడిల్‌వేర్ లేకుండా** (14-conditional-workflow.ipynb):
- పారిస్ వద్ద గదులు లేవు → alternative_agent కు రూటు వేయడం

**మిడిల్‌వేర్‌తో** (ఈ నోట్బుక్):
- సాధారణ వినియోగదారు + పారిస్ → గదులు లేవు → alternative_agent కు రూట్
- ప్రాధాన్య వినియోగదారు + పారిస్ → 🌟 మిడిల్‌వేర్ ఓవర్రైడ్స్! → అందుబాటులో ఉంది → booking_agent కు రూట్

## అగ్రిమెంట్ల జాబితా:
- Microsoft Agent Framework ఇన్‌స్టాల్ చేయబడింది
- కండిషనల్ వర్క్‌ఫ్లోలను అర్థం చేసుకోవడం (చూపండి 14-conditional-workflow.ipynb)
- GitHub టోకెన్ లేదా OpenAI API కీ
- మిడిల్‌వేర్ ప్యాటర్న్స్ గురించి ప్రాథమిక అవగాహన


In [1]:
import asyncio
import json
import os
from collections.abc import Awaitable, Callable
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    ChatMessage,
    FunctionInvocationContext,
    Role,
    WorkflowBuilder,
    WorkflowContext,
    ai_function,
    executor,
)

# 🤖 GitHub Models or OpenAI client integration
from agent_framework.openai import OpenAIChatClient
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")

✅ All imports successful!


## Step 1: నిర్మిత అవుట్పుట్ల కోసం Pydantic మోడల్స్ నిర్వచించండి

ఈ మోడల్స్ ఏజెంట్లు సహా **స్కీమా**ని నిర్వచిస్తాయి. మేము ఒక `priority_override` ఫీల్డ్‌ను యాడ్ చేశాము, ఇది మీడియావేర్ లభ్యత ఫలితాన్ని మార్పు చేస్తున్నప్పుడు ట్రాక్ చేయడానికి.


In [2]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str
    priority_override: bool = False  # 🆕 NEW! Tracks if middleware overrode the result


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check with priority_override)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

✅ Pydantic models defined:
   - BookingCheckResult (availability check with priority_override)
   - AlternativeResult (alternative suggestion)
   - BookingConfirmation (booking confirmation)


## దశ 2: ప్రాధాన్యత సభ్యుల డేటాబేస్ నిర్వచించండి

ఈ డెమో కోసం, మేము ఒక ప్రాధాన్యత సభ్యత డేటాబేస్‌ను అనుకరించబోతున్నాము. ఉత్పత్తిలో, ఇది నిజమైన డేటాబేస్ లేదా APIని ప్రశ్నిస్తుంది.

**ప్రాధాన్యత సభ్యులు:**
- `alice@example.com` - VIP సభ్యుడు
- `bob@example.com` - ప్రీమియం సభ్యుడు  
- `priority_user` - పరీక్ష ఖాతా


In [3]:
# Simulated priority members database
PRIORITY_MEMBERS = {
    "alice@example.com",
    "bob@example.com",
    "priority_user",
}

# Global variable to track current user (in real app, use proper session management)
current_user_id = "regular_user"  # Default: regular user


def set_user(user_id: str):
    """Set the current user for the session."""
    global current_user_id
    current_user_id = user_id
    is_priority = user_id in PRIORITY_MEMBERS
    status = "🌟 PRIORITY MEMBER" if is_priority else "👤 Regular User"

    display(
        HTML(f"""
        <div style='padding: 15px; background: {"linear-gradient(135deg, #FFD700 0%, #FFA500 100%)" if is_priority else "#e3f2fd"}; 
                    border-left: 4px solid {"#FF6B35" if is_priority else "#2196f3"}; border-radius: 4px; margin: 10px 0;'>
            <strong>👤 Current User Set:</strong> {user_id}<br>
            <strong>Status:</strong> {status}
        </div>
    """)
    )


print("✅ Priority members database created")
print(f"   Priority members: {len(PRIORITY_MEMBERS)} users")

✅ Priority members database created
   Priority members: 3 users


## Step 3: హోటల్ బుకింగ్ సాధనం సృష్టించండి

కండిషనల్ వర్క్‌ఫ్లో లా ఉంటుంది, కానీ ఇప్పుడు ఇది మిడిల్వేర్ ద్వారా అడ్డుకోబడుతుంది!


In [4]:
@ai_function(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.
    
    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @ai_function decorator")

✅ hotel_booking tool created with @ai_function decorator


## Step 4: 🌟 ప్రాధాన్యత దృశ్యమానం మధ్యవర్తి (ముఖ్య లక్షణం!)

ఇది ఈ నోట్‌బుక్ యొక్క **కోర్ ఫంక్షనాలిటీ**. మధ్యవర్తి:

1. హోటల్_బుకింగ్ ఫంక్షన్ కాల్‌ను **అడ్డుకుంటుంది**
2. `next(context)` ని పిలవడం ద్వారా ఫంక్షన్‌ను సాధారణంగా **నిర్వహిస్తుంది**
3. `context.result` లో ఫలితాన్ని **పరిశీలిస్తుంది**
4. వినియోగదారుడు ప్రాధాన్యత ఉంటే మరియు గదులు అందుబాటులో లేకపోతే ఫలితాన్ని **మెరుగుపరుస్తుంది**
5. కవరించిన ఫలితాన్ని ఏజెంట్‌కు **తిరిగి అందిస్తుంది**

**ప్రధాన నమూనా:**
```python
async def my_middleware(context, next):
    await next(context)  # ఫంక్షన్ ను అమలు చేయండి
    # ఇప్పుడు context.result లో ఫంక్షన్ యొక్క ఫలితం ఉంటుంది
    if some_condition:
        context.result = new_value  # మించు!
```


In [5]:
async def priority_check_middleware(
    context: FunctionInvocationContext,
    next: Callable[[FunctionInvocationContext], Awaitable[None]],
) -> None:
    """
    Function middleware that overrides hotel_booking results for priority members.
    
    Workflow:
    1. Let the function execute normally
    2. Check if user is a priority member
    3. If priority + no availability → Override to make rooms available!
    4. Agent will then route to booking path instead of alternative path
    """
    function_name = context.function.name

    display(
        HTML(f"""
        <div style='padding: 12px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
            <strong>🔄 Middleware:</strong> Intercepting {function_name}...
        </div>
    """)
    )

    # Execute the original function
    await next(context)

    # Now inspect and potentially modify the result
    if context.result and function_name == "hotel_booking":
        result_data = json.loads(context.result)
        destination = result_data.get("destination", "")
        has_availability = result_data.get("has_availability", False)

        # Check if user is priority member
        is_priority = current_user_id in PRIORITY_MEMBERS

        # Override logic: Priority member + no availability → Make available!
        if is_priority and not has_availability:
            display(
                HTML(f"""
                <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); 
                            border-radius: 8px; margin: 10px 0; box-shadow: 0 4px 12px rgba(255,165,0,0.4);'>
                    <h3 style='margin: 0 0 10px 0; color: #333;'>🌟 PRIORITY OVERRIDE ACTIVATED! 🌟</h3>
                    <p style='margin: 0; color: #555; line-height: 1.6;'>
                        <strong>User:</strong> {current_user_id}<br>
                        <strong>Status:</strong> VIP Priority Member<br>
                        <strong>Action:</strong> Overriding "No Availability" for {destination}<br>
                        <strong>Result:</strong> ✅ Rooms now available for priority booking!
                    </p>
                </div>
            """)
            )

            # Override the result!
            result_data["has_availability"] = True
            result_data["priority_override"] = True
            context.result = json.dumps(result_data)

        elif not has_availability:
            display(
                HTML(f"""
                <div style='padding: 12px; background: #ffebee; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>ℹ️ Middleware:</strong> No priority override (user: {current_user_id})
                </div>
            """)
            )


print("✅ priority_check_middleware created")
print("   - Intercepts hotel_booking function")
print("   - Overrides availability for priority members")

✅ priority_check_middleware created
   - Intercepts hotel_booking function
   - Overrides availability for priority members


## Step 5: రూటింగ్ కోసం పరిస్థితి ఫంక్షన్లను నిర్వచించండి

సమాన పరిస్థితి ఫంక్షన్లు కండిషనల్ వర్క్ఫ్లోలా ఉంటాయి - అవి నిర్మిత అవుట్పుట్‌ను పరిశీలించి రూటింగ్‌ను నిర్ణయిస్తాయని.


In [6]:
def has_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels ARE available (including priority overrides!)."""
    if not isinstance(message, AgentExecutorResponse):
        return True

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        # Check if this was a priority override
        override_indicator = " 🌟" if result.priority_override else ""

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}{override_indicator}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels are NOT available."""
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception:
        return False


print("✅ Condition functions defined")

✅ Condition functions defined


## Step 6: కస్టమ్ డిస్‌ప్లే ఎగ్జిక్యూటర్ సృష్టించండి

మునుపటి ఎగ్జిక్యూటర్ మాదిరిగానే - తుదిశాఖ వర్క్‌ఫ్లో అవుట్‌పుట్‌ను ప్రదర్శిస్తుంది.


In [7]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """Display the final result as workflow output."""
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created")

✅ display_result executor created


## Step 7: ఇంటర్వ్యూమెంటు వేరియబుల్స్ లోడ్ చేయండి

LLM క్లయింట్ (GitHub మోడల్స్ లేదా OpenAI) ను కాన్ఫిగర్ చేయండి.


In [8]:
# Load environment variables
load_dotenv()

# Check for GitHub Models or OpenAI
chat_client = OpenAIChatClient(base_url=os.environ.get(
    "GITHUB_ENDPOINT"), api_key=os.environ.get("GITHUB_TOKEN"), model_id="gpt-4o")


## స్టెప్ 8: మిడిల్‌వేర్‌తో AI ఏజెంట్లను సృష్టించండి

**ముఖ్య తేడా:** availability_agent ను సృష్టిస్తున్నప్పుడు, మనం `middleware` పరామితిని అందిస్తున్నాము!

ఇదే విధంగా మేం priority_check_middleware ను ఏజెంట్ యొక్క ఫంక్షన్ల ఆహ్వాన పైప్లైన్లో ఇంజెక్ట్ చేస్తున్నాము.


In [9]:
# Agent 1: Check availability with tool + middleware
availability_agent = AgentExecutor(
    chat_client.create_agent(
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), message (string), "
            "and priority_override (bool, true if priority member got special access). "
            "The message should summarize the availability status and mention if priority override occurred."
        ),
        tools=[hotel_booking],
        response_format=BookingCheckResult,
        middleware=[priority_check_middleware],  # 🌟 MIDDLEWARE INJECTION!
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    chat_client.create_agent(
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        response_format=AlternativeResult,
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    chat_client.create_agent(
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "If priority_override is true in the input, mention that they received priority member access. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        response_format=BookingConfirmation,
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - WITH priority_check_middleware 🌟</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)

## Step 9: పని ప్రవాహాన్ని నిర్మించండి

మునుపటి తరహాలోనే పని ప్రవాహ నిర్మాణం - అందుబాటుపై ఆధారపడి పరిస్థితి ఆధారిత మార్గ నిర్దేశం.


In [10]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder()
    .set_start_executor(availability_agent)
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH (can be triggered by middleware override!)
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing (Middleware-Aware):</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> (or 🌟 <strong>priority override</strong>) → booking_agent → display_result
        </p>
    </div>
""")
)

## Step 10: పరీక్ష కేసు 1 - పారిస్ లో రెగ్యులర్ యూజర్ (ఓవర్‌రైడ్ లేదు)

ఒక రెగ్యులర్ యూజర్ పారిస్ బుక్ చేయడానికి ప్రయత్నిస్తుంది → గది లేవు → alternative_agent కు మార్గాలు


In [11]:
# Set as regular user
set_user("regular_user")

display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Regular User + Paris</h3>
        <p style='margin: 0;'><strong>Expected:</strong> No rooms → No middleware override → Alternative suggestion</p>
    </div>
""")
)

# Create request
request_regular = AgentExecutorRequest(
    messages=[ChatMessage(Role.USER, text="I want to book a hotel in Paris")], should_respond=True
)

# Run workflow
events_regular = await workflow.run(request_regular)
outputs_regular = events_regular.get_outputs()

# Display results
if outputs_regular:
    result_regular = AlternativeResult.model_validate_json(outputs_regular[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: #fff; border: 2px solid #ff9800; border-radius: 12px; margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #e65100;'>📊 RESULT (Regular User)</h3>
            <div style='background: #fff3e0; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0;'><strong>Middleware:</strong> No priority override (regular user)</p>
                <p style='margin: 0 0 10px 0;'><strong>Alternative:</strong> 🏨 {result_regular.alternative_destination}</p>
                <p style='margin: 0;'><strong>Reason:</strong> {result_regular.reason}</p>
            </div>
        </div>
    """)
    )

## Step 11: టెస్ట్ కేస్ 2 - 🌟 ప్రాధాన్యత వినియోగదారు పారిస్ లో (ఓవర్‌రైడ్‌తో!)

ఒక ప్రాధాన్యత సభ్యుడు పారిస్ బుక్ చేయడానికి ప్రయత్నిస్తాడు → ప్రారంభంలో తరగతులు లేవు → 🌟 మిడిల్‌వేర్ ఓవర్‌రైడ్ చేస్తుంది! → booking_agent కు మార్గం

**ఇది మిడిల్‌వేర్ శక్తి యొక్క కీలక ప్రదర్శన!**


In [12]:
# Set as priority user
set_user("priority_user")

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #333;'>🧪 TEST CASE 2: 🌟 Priority User + Paris</h3>
        <p style='margin: 0; color: #555;'><strong>Expected:</strong> No rooms → 🌟 MIDDLEWARE OVERRIDE → Rooms available → Booking suggestion!</p>
    </div>
""")
)

# Create request
request_priority = AgentExecutorRequest(
    messages=[ChatMessage(Role.USER, text="I want to book a hotel in Paris")], should_respond=True
)

# Run workflow
events_priority = await workflow.run(request_priority)
outputs_priority = events_priority.get_outputs()

# Display results
if outputs_priority:
    result_priority = BookingConfirmation.model_validate_json(outputs_priority[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; 
                    box-shadow: 0 8px 16px rgba(255,165,0,0.4); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 RESULT (Priority Member) 🌟</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Priority Override!)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> 🌟 OVERRIDE ACTIVATED!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_priority.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_priority.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_priority.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #fff3cd; border-radius: 6px; border-left: 4px solid #FF6B35;'>
                    <strong>💡 What Just Happened:</strong><br>
                    1. hotel_booking tool returned "no availability"<br>
                    2. priority_check_middleware intercepted the result<br>
                    3. Middleware checked user status: priority_user ✅<br>
                    4. Middleware OVERRODE the result to "available"<br>
                    5. Workflow routed to booking_agent instead of alternative_agent!
                </div>
            </div>
        </div>
    """)
    )

## Step 12: టెస్ట్ కేస్ 3 - ప్రాధాన్యత వినియోగదారు స్టాక్‌హోమ్‌లో (ఇప్పటికే అందుబాటులో ఉంది)

ప్రాధాన్యత వినియోగదారు స్టాక్‌హోమ్‌ను ప్రయత్నిస్తారు → గదులు అందుబాటులో ఉన్నాయి → ఎలాంటి ఓవర్‌రైడ్ అవసరం లేదు → booking_agent కు మార్గనిర్దేశం చేస్తుంది

ఇది మిడిల్‌వేర్ అవసరమయినప్పుడు మాత్రమే చర్య తీసుకుంటుందని చూపిస్తుంది!


In [13]:
# Priority user is still set from previous test

display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 3: Priority User + Stockholm</h3>
        <p style='margin: 0;'><strong>Expected:</strong> Rooms available → No override needed → Booking suggestion</p>
    </div>
""")
)

# Create request
request_stockholm = AgentExecutorRequest(
    messages=[ChatMessage(Role.USER, text="I want to book a hotel in Stockholm")], should_respond=True
)

# Run workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; 
                    box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 RESULT (Priority User - No Override Needed)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Natural)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> No override needed</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #e8f5e9; border-radius: 6px; border-left: 4px solid #4caf50;'>
                    <strong>💡 Middleware Behavior:</strong><br>
                    • hotel_booking returned "available" naturally<br>
                    • Middleware saw available = true → No override needed<br>
                    • Workflow proceeded normally to booking_agent
                </div>
            </div>
        </div>
    """)
    )

## ముఖ్యమైన విషయాలు మరియు మిడిల్వేర్ భావనలు

### ✅ మీరు నేర్చుకున్నది:

#### **1. ఫంక్షన్-ఆధారిత మిడిల్వేర్ నమూనా**

మిడిల్వేర్ సాదా async ఫంక్షన్ ఉపయోగించి ఫంక్షన్ కాల్స్‌ను ఇంటర్‌సెప్టు చేస్తుంది:

```python
async def my_middleware(
    context: FunctionInvocationContext,
    next: Callable,
) -> None:
    # ఫంక్షన్ నిర్వహణకు ముందు
    print("Intercepting...")
    
    # ఫంక్షన్ నిర్వహించండి
    await next(context)
    
    # ఫంక్షన్ నిర్వహణకి తరువాత - ఫలితాన్ని పరిశీలించండి
    if context.result:
        # అవసరమైతే ఫలితాన్ని మార్చండి
        context.result = modified_value
```

#### **2. కాంటెక్స్ట్ యాక్సెస్ మరియు ఫలిత ఓవర్‌రైడ్**

- `context.function` - పిలవబడుతోన్న ఫంక్షన్‌ను యాక్సెస్ చేయండి  
- `context.arguments` - ఫంక్షన్ ఆర్గ్యుమెంట్లను చదవండి  
- `context.kwargs` - అదనపు పారామీటర్లకు యాక్సెస్  
- `await next(context)` - ఫంక్షన్‌ను నడపండి  
- `context.result` - ఫంక్షన్ అవుట్‌పుట్‌ను చదవండి/మార్చండి  

#### **3. వ్యాపార లాజిక్ అమలు**

మన మిడిల్వేర్ ప్రాధాన్యత సభ్య లాభాలను అమలు చేస్తుంది:  
- **నిబంధిత యూజర్స్**: మార్పులు అవసరం లేదు, ప్రాథమిక వర్క్‌ఫ్లో  
- **ప్రాధాన్యత యూజర్స్**: "లಭ್ಯత లేదు" → "లభ్యం"ను ఓవర్‌రైడ్ చేస్తుంది  
- **షరతు లాజిక్**: అవసరమైతే మాత్రమే ఓవర్‌రైడ్ చేస్తుంది  

#### **4. అదే వర్క్‌ఫ్లో, భిన్న ఫలితాలు**

మిడిల్వేర్ శక్తి:  
- ✅ వర్క్‌ఫ్లో నిర్మాణంలో మార్పులు లేవు  
- ✅ టూల్ ఫంక్షన్‌లో మార్పులు లేవు  
- ✅ షరతు రౌటింగ్ లాజిక్‌లో మార్పులు లేవు  
- ✅ కేవలం మిడిల్వేర్ → వివిధ ప్రవర్తన!

### 🚀 నిజమైన ప్రపంచ అనువర్తనాలు:

1. **VIP/ప్రీమియం ఫీచర్లు**  
   - ప్రీమియం యూజర్స్ కోసం రేట్ పరిమితులను ఓవర్‌రైడ్ చేయండి  
   - రిసోర్సులకు ప్రాధాన్యత యాక్సెస్ అందించండి  
   - ప్రీమియం ఫీచర్లను డైనమిక్‌గా అన్‌లాక్ చేయండి  

2. **A/B టెస్టింగ్**  
   - వినియోగదారులను వేర్వేరు అమలులకు మార్గనిర్దేశం చేయండి  
   - కొత్త ఫీచర్లను నిర్దిష్ట యూజర్స్‌తో పరీక్షించండి  
   - గమనోపయోగ ఫీచర్ విడుదలలు  

3. **సెక్యూరిటీ & కంప్లయిన్స్**  
   - ఫంక్షన్ కాల్స్‌ను అడిట్ చేయండి  
   - సంసిద్ధ ఆపరేషన్లను బ్లాక్ చేయండి  
   - వ్యాపార నిబంధనలను అమలు చేయండి  

4. **పర్ఫార్మెన్స్ ఆప్టిమైజేషన్**  
   - నిర్దిష్ట యూజర్స్ కోసం ఫలితాలను క్యాష్ చేయండి  
   - ఖర్చైన ఆపరేషన్లను సంభవించేటప్పుడు తప్పించవచ్చు  
   - డైనమిక్ రిసోర్స్ కేటాయింపు  

5. **లోపాల నిర్వహణ & రీట్రై**  
   - లోపాలను అందంగా క్యాచ్ చేసి హ్యాండిల్ చేయండి  
   - రీట్రై లాజిక్‌ను అమలు చేయండి  
   - ప్రత్యామ్నాయ అమలుకు fallback  

6. **లాగింగ్ & మానిటరింగ్**  
   - ఫంక్షన్ నడిచే సమయాలను ట్రాక్ చేయండి  
   - ఆర్గ్యుమెంట్లు మరియు ఫలితాలను లాగ్ చేయండి  
   - వినియోగ నమూనాలను మానిటర్ చేయండి  

### 🔑 డెకరేటర్ల నుండి ముఖ్యమైన తేడాలు:

| ఫీచర్ | డెకరేటర్ | మిడిల్వేర్ |
|---------|-----------|------------|
| **విస్తృతి** | ఒకే ఫంక్షన్ | ఏజెంట్‌లోని అన్ని ఫంక్షన్లు |
| **లవచీకరణ** | నిర్వచన సమయంలో స్థిరంగా | రన్‌టైమ్‌లో డైనమిక్‌గా |
| **కాంటెక్స్ట్** | పరిమితం | పూర్తి ఏజెంట్ కాంటెక్స్ట్ |
| **కంపోజిషన్** | బహుళ డెకరేటర్లు | మిడిల్వేర్ పైప్‌లైన్ |
| **ఏజెంట్-అవేర్** | కాదు | అవును (ఏజెంట్ స్టేట్ యాక్సెస్) |

### 📚 ఎప్పుడు మిడిల్వేర్ ఉపయోగించాలి:

✅ **మిడిల్వేర్ ఉపయోగించండి:**
- వినియోగదారుడు/సెషన్ స్టేట్ ఆధారంగా ప్రవర్తనను మార్చాలి  
- అనేక ఫంక్షన్లపై లాజిక్ వర్తింప చేయాలి  
- ఏజెంట్ స్థాయి కాంటెక్స్ట్‌ను యాక్సెస్ చేయాలి  
- క్రాస్-కట్టింగ్ అంశాలు (లాగింగ్, ఆథ్, ఇత్యాది) అమలు చేయాలి  

❌ **మిడిల్వేర్ ఉపయోగించకండి:**
- సింపుల్ ఇన్‌పుట్ వెరిఫికేషన్ (Pydantic ఉపయోగించండి)  
- ఫంక్షన్-ప్రత్యేక లాజిక్ (ఫంక్షన్ లో ఉంచండి)  
- ఒకసారి మార్పులు (ఫంక్షన్ మార్చడం సరిపోతుంది)  

### 🎓 అభివృద్ధి చెందిన నమూనాలు:

```python
# ఒకাধিক మధ్యవర్తి (నిర్వహణ క్రమం ముఖ్యం!)
middleware=[
    logging_middleware,      # మొదట లాగ్స్
    auth_middleware,         # ఆ తరువాత అథారిటీ తనిఖీ చేస్తుంది
    cache_middleware,        # ఆ తరువాత క్యాష్ తనిఖీ చేస్తుంది
    rate_limit_middleware,   # ఆ తరువాత రేటు పరిమితులు
    priority_check_middleware  # చివరికి ప్రాధાન્ય తనిఖీ
]

# శరతామూల మధ్యవర్తి నిర్వహణ
async def conditional_middleware(context, next):
    if should_execute(context):
        await next(context)
        # ఫలితాన్ని మార్చండి
    else:
        # పూర్తిగా నిర్వహణ తప్పించండి
        context.result = cached_value
```

### 🔗 సంబంధించిన భావనలు:

- **ఏజెంట్ మిడిల్వేర్**: agent.run() కాల్స్‌ను ఇంటర్‌సెప్టు చేస్తుంది  
- **ఫంక్షన్ మిడిల్వేర్**: టూల్ ఫంక్షన్ కాల్స్‌ను ఇంటర్‌సెప్టు చేస్తుంది (మనం ఉపయోగించినది!)  
- **మిడిల్వేర్ పైప్‌లైన్**: క్రమంగా అమలుచేసే మిడిల్వేర్ గొలుసు  
- **కాంటెక్స్ట్ ప్రొపగేషన్**: మిడిల్వేర్ గొలుసు ద్వారా స్టేట్ వాహనం


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**స్పష్టం**:  
ఈ పత్రాన్ని AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించారు. మేము సరైన అనువాదానికి ప్రయత్నించినప్పటికీ, ఆటోమేటెడ్ అనువాదాలలో లోపాలు లేదా తప్పుపాట్లు ఉండవచ్చు. ఆ కుటుంబ భాషలోని అసలైన పత్రం అధికారిక మూలంగా పరిగణించాలి. కీలక సమాచారానికి, ప్రొఫెషనల్ మానవ అనువాదం సిఫార్సు చేయబడింది. ఈ అనువాదం ఉపయోగం వల్ల కలిగే ఏవైనా అర్థం లోపాలు లేదా తప్పుడు అర్థవ్యాఖ్యలకు మేము బాధ్యత వహించము.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
